In [109]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [110]:
#read csv data into pandas, and initial data exploration
df = pd.read_csv('housing_dirty.csv')
df.head(10)



,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
0,1,297.0,1084.0,jogja,2.0,2000,baik
1,2,254.0,761.0,Medan,NaN,1995,Bagus
2,3,249.7,895.0,Depok,NaN,1983,baik
3,4,49.7,178.0,YGY,5.0,2013,baik
4,5,133.4,424.0,Medan,5.0,2004,Sedang
5,6,153.3,814.0,Jakarta,1.0,2006,Sedang
6,7,114.3,NaN,jakarta,3.0,2011,baik
7,8,NaN,333.0,Yogyakarta,NaN,1989,baik sekali
8,9,81.2,307.0,Yogyakarta,1.0,1996,SEDANG
9,10,69.1,237.0,Bandung,5.0,1980,baik


In [111]:
#Initial Data Exploration
## Column Information, and Data Type
print('\n')
df.info()

df.describe()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.2+ KB


,id,luas_m2,harga_juta,kamar,tahun_bangun
count,130.000000,112.000000,1.130000e+02,120.000000,130.000000
mean,65.500000,267.627679,8.856325e+05,3.433333,2062.638462
std,37.671829,885.664181,9.407144e+06,1.776283,701.684043
min,1.000000,-50.000000,-5.000000e+02,1.000000,1890.000000
25%,33.250000,87.050000,3.450000e+02,2.000000,1991.250000
50%,65.500000,193.800000,6.550000e+02,4.000000,2002.000000
75%,97.750000,280.675000,9.550000e+02,5.000000,2011.750000
max,130.000000,9500.000000,1.000000e+08,6.000000,9999.000000


In [ ]:
## Check is there any null value inside column
print('\n Missing Values')
totalColumnWithNullValues = df.isnull().sum()
print(totalColumnWithNullValues)

## Percentage of missing values
print('\nPercentage of Missing Values')
percentageOfMissingValue = ((totalColumnWithNullValues/len(df))*100).round(2)
print(percentageOfMissingValue)


 Missing Values
id               0
luas_m2         18
harga_juta      17
kota             0
kamar           10
tahun_bangun     0
kondisi          0
dtype: int64

 Percentage of Missing Values
id               0.00
luas_m2         13.85
harga_juta      13.08
kota             0.00
kamar            7.69
tahun_bangun     0.00
kondisi          0.00
dtype: float64


In [113]:
## Checking and Removing Duplicated Values
isDataframeHasAnyDuplicates = df.duplicated().sum() >= 1
if isDataframeHasAnyDuplicates:
    df.drop_duplicates(inplace=True) 
else:
    print("Dataframe has no duplicated rows")

Dataframe has no duplicated rows


In [114]:
## String Normalization
df['kota'] = df['kota'].str.strip().str.title()
df['kondisi'] = df['kondisi'].str.strip().str.lower()

df.head(20)

,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
0,1,297.0,1084.0,Jogja,2.0,2000,baik
1,2,254.0,761.0,Medan,NaN,1995,bagus
2,3,249.7,895.0,Depok,NaN,1983,baik
3,4,49.7,178.0,Ygy,5.0,2013,baik
4,5,133.4,424.0,Medan,5.0,2004,sedang
5,6,153.3,814.0,Jakarta,1.0,2006,sedang
6,7,114.3,NaN,Jakarta,3.0,2011,baik
7,8,NaN,333.0,Yogyakarta,NaN,1989,baik sekali
8,9,81.2,307.0,Yogyakarta,1.0,1996,sedang
9,10,69.1,237.0,Bandung,5.0,1980,baik


In [115]:
## Missing values imputation
## Check if the value is skewed, if so we do median, else we do the mean/average
df['luas_m2'] = df['luas_m2'].fillna(df['luas_m2'].median())
df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median()) 
 

## Fill in missing categorical(string) with the most appeared data in the dataframe (modus)
df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])


##Check for any missing values
print('\nMissing Values')
totalColumnWithNullValues = df.isnull().any()
print(totalColumnWithNullValues)



Missing Values
id              False
luas_m2         False
harga_juta      False
kota            False
kamar           False
tahun_bangun    False
kondisi         False
dtype: bool


In [116]:
## Check Outlier, sampling one with maximum harga juta
print(df.loc[df['harga_juta'].idxmax()])

##Handling Outlier Data using IQR Fence Method
for column in ['harga_juta','luas_m2','tahun_bangun']:
    Q1,Q3 = df[column].quantile([0.25,0.75])
    IQR = Q3 - Q1
    df[column] = df[column].clip(Q1-1.5*IQR, Q3+1.5*IQR)

id                     102
luas_m2              221.6
harga_juta      99999999.0
kota                 Medan
kamar                  4.0
tahun_bangun          2022
kondisi             sedang
Name: 101, dtype: object


In [117]:
## Sampling row with maximum harga juta 
df[df['id'] == 102]
## Hereby we see that the harga_juta is updated from 99999999.0 to 1719.25

,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
101,102,221.6,1719.25,Medan,4.0,2022.0,sedang


In [118]:
## Assert Condition
assert df.isnull().sum().sum() == 0, 'Masih Ada Data Missing'
assert df.duplicated().sum().sum() == 0, 'Masih Ada Data Duplicate'

In [119]:
## Extra Task, Get Response from JSONPlaceHolder and returns as dataframe
import requests, pandas as pd
from pandas import json_normalize
from IPython.display import display

url = "https://jsonplaceholder.typicode.com/users"

resp = requests.get(url)
if resp.status_code == 200:
    data = resp.json()
    df = json_normalize(data, sep='_')
    display(df.head())
else:
    print(f'Cannot Access the Url with the following status code: {resp.status_code}')


,id,name,username,email,phone,website,address_street,address_suite,address_city,address_zipcode,address_geo_lat,address_geo_lng,company_name,company_catchPhrase,company_bs
0,1,Leanne Graham,Bret,Sincere@april.biz,1-770-736-8031 x56442,hildegard.org,Kulas Light,Apt. 556,Gwenborough,92998-3874,-37.3159,81.1496,Romaguera-Crona,Multi-layered client-server neural-net,harness real-time e-markets
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,010-692-6593 x09125,anastasia.net,Victor Plains,Suite 879,Wisokyburgh,90566-7771,-43.9509,-34.4618,Deckow-Crist,Proactive didactic contingency,synergize scalable supply-chains
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,1-463-123-4447,ramiro.info,Douglas Extension,Suite 847,McKenziehaven,59590-4157,-68.6102,-47.0653,Romaguera-Jacobson,Face to face bifurcated interface,e-enable strategic applications
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,493-170-9623 x156,kale.biz,Hoeger Mall,Apt. 692,South Elvis,53919-4257,29.4572,-164.2990,Robel-Corkery,Multi-tiered zero tolerance productivity,transition cutting-edge web services
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,(254)954-1289,demarco.info,Skiles Walks,Suite 351,Roscoeview,33263,-31.8129,62.5342,Keebler LLC,User-centric fault-tolerant solution,revolutionize end-to-end systems
